In [ ]:
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
# CEL·LA 0 — Executa PRIMER aquesta cel·la
# Fixa les versions compatibles amb el model Helsinki/opus-mt-es-ca
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
import subprocess, sys

paquets = [
    'transformers==4.45.0',
    'sentencepiece',
    'sacremoses',
    'evaluate',
    'sacrebleu',
    'datasets',
    'accelerate',
]

print('Instal·lant dependències...')
for pkg in paquets:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet'],
        capture_output=True
    )
    marca = '\u2713' if r.returncode == 0 else '\u2717'
    print(f'  {marca} {pkg}')

# Verifica importacions crítiques
errors = []
for m in ['transformers', 'datasets', 'evaluate', 'sacrebleu', 'sentencepiece']:
    try:
        __import__(m)
        print(f'  \u2713 {m}')
    except ImportError:
        errors.append(m)
        print(f'  \u2717 {m}')

if errors:
    print(f'\n\u26a0 M\u00f2duls que falten: {errors}')
    print('Reinicia la sessi\u00f3 i torna a executar.')
else:
    import transformers
    assert transformers.__version__.startswith('4.'), \
        f'Versi\u00f3 incorrecta: {transformers.__version__}'
    print(f'\n\u2713 Transformers {transformers.__version__} \u2014 tot llest.')


# Afinament motor TAN SLPL·UV
## Model base: Helsinki-NLP/opus-mt-es-ca → domini UV
### Servei de Llenagües i Política Lingüística · Universitat de València

> ⚠️ **IMPORTANT**: Executa PRIMER la cel·la 0. Després: **Entorn d'execució → Executa-ho tot**

### Estructura del corpus a Google Drive
```
MyDrive/SLPL_TAN/
├── corpus/
│   ├── train.es  (frases castellà)
│   ├── train.ca  (frases valencià)
│   ├── val.es    (frases castellà validació)
│   └── val.ca    (frases valencià validació)
└── model-afinar/  (model guardat automàticament)
```


In [ ]:
import os, sys, pathlib

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

IS_KAGGLE = os.path.exists('/kaggle') and not IS_COLAB
print(f'Entorn: {"Colab" if IS_COLAB else "Kaggle" if IS_KAGGLE else "Local"}')

if IS_COLAB:
    from google.colab import drive
    print('Muntant Drive...')
    try:
        drive.mount('/content/drive', force_remount=True)
        print('\u2713 Drive muntat.')
    except Exception as e:
        print(f'\u26a0 Error: {e}')
        raise
    CORPUS_DIR = '/content/drive/MyDrive/SLPL_TAN/corpus'
    OUTPUT_DIR = '/content/drive/MyDrive/SLPL_TAN/model-afinar'
elif IS_KAGGLE:
    CORPUS_DIR = '/kaggle/input/slpl-corpus'
    OUTPUT_DIR = '/kaggle/working/model-afinar'
else:
    CORPUS_DIR = r'C:\Users\santi\OneDrive\Documents\SLPL\taneu\corpus d entrenament i afinament\processed'
    OUTPUT_DIR = r'C:\Users\santi\OneDrive\Documents\SLPL\taneu\model-afinar'

pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Verifica corpus
corpus_ok = all(
    os.path.exists(os.path.join(CORPUS_DIR, f))
    for f in ['train.es', 'train.ca', 'val.es', 'val.ca']
)
if not corpus_ok:
    fitxers = os.listdir(CORPUS_DIR) if os.path.exists(CORPUS_DIR) else []
    print(f'\u26a0 Fitxers trobats: {fitxers}')
    raise FileNotFoundError(f'Falten fitxers del corpus a {CORPUS_DIR}')

print(f'\u2713 Corpus verificat.')
print(f'  Corpus:  {CORPUS_DIR}')
print(f'  Sortida: {OUTPUT_DIR}')


In [ ]:
import torch

cuda_ok = torch.cuda.is_available()
print(f'CUDA: {cuda_ok}')
if cuda_ok:
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
    DEVICE, FP16, BATCH = 'cuda', True, 16
else:
    print('\u26a0 Sense GPU.')
    DEVICE, FP16, BATCH = 'cpu', False, 4


In [ ]:
def carrega_fitxer(path):
    with open(path, encoding='utf-8') as f:
        return [l.strip() for l in f if l.strip()]

print('Carregant corpus...')
train_src = carrega_fitxer(os.path.join(CORPUS_DIR, 'train.es'))
train_tgt = carrega_fitxer(os.path.join(CORPUS_DIR, 'train.ca'))
val_src   = carrega_fitxer(os.path.join(CORPUS_DIR, 'val.es'))
val_tgt   = carrega_fitxer(os.path.join(CORPUS_DIR, 'val.ca'))

train_pairs = list(zip(train_src, train_tgt))
val_pairs   = list(zip(val_src,   val_tgt))

print(f'\u2713 Corpus carregat:')
print(f'  Train: {len(train_pairs)} parells')
print(f'  Val:   {len(val_pairs)} parells')

# Mostra 3 exemples
import random
random.seed(42)
print('\n3 exemples aleatoris:')
for s, t in random.sample(train_pairs, 3):
    print(f'  ES: {s[:80]}')
    print(f'  CA: {t[:80]}')
    print()


In [ ]:
from transformers import MarianMTModel, MarianTokenizer

MODEL_ID = 'Helsinki-NLP/opus-mt-es-ca'
print(f'Carregant: {MODEL_ID}')

tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
model     = MarianMTModel.from_pretrained(MODEL_ID)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'\u2713 Model carregat. Par\u00e0metres: {params:.1f}M')

# Prova base
test = 'El servicio de lenguas ofrece asesoramiento ling\u00fc\u00edstico.'
inputs = tokenizer([test], return_tensors='pt', padding=True)
output = model.generate(**inputs)
print(f'\nProva base:')
print(f'  ES: {test}')
print(f'  CA: {tokenizer.decode(output[0], skip_special_tokens=True)}')


In [ ]:
from datasets import Dataset

MAX_LEN = 256

def preprocess(examples):
    sources = [ex['es'] for ex in examples['translation']]
    targets = [ex['ca'] for ex in examples['translation']]
    model_inputs = tokenizer(
        sources, max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_LEN, truncation=True, padding='max_length'
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in label]
        for label in labels['input_ids']
    ]
    return model_inputs

train_dataset = Dataset.from_dict(
    {'translation': [{'es': s, 'ca': t} for s, t in train_pairs]}
)
val_dataset = Dataset.from_dict(
    {'translation': [{'es': s, 'ca': t} for s, t in val_pairs]}
)

print('Tokenitzant...', end=' ')
train_tok = train_dataset.map(
    preprocess, batched=True, num_proc=2, remove_columns=['translation']
)
val_tok = val_dataset.map(
    preprocess, batched=True, num_proc=2, remove_columns=['translation']
)
train_tok.set_format('torch')
val_tok.set_format('torch')
print('fet.')
print(f'\u2713 Train: {len(train_tok)} | Val: {len(val_tok)}')


In [ ]:
print('Verificaci\u00f3 pr\u00e8via:')
print(f'  Dispositiu:    {DEVICE}')
print(f'  FP16:          {FP16}')
print(f'  Batch size:    {BATCH}')
print(f'  Gradient acc:  2')
print(f'  Max length:    {MAX_LEN}')
print(f'  Train parells: {len(train_tok)}')
print(f'  Val parells:   {len(val_tok)}')

if DEVICE == 'cuda':
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  VRAM lliure:   {mem:.1f} GB')

steps_per_epoch = len(train_tok) // (BATCH * 2)
temps_h = steps_per_epoch * 3 * 1.5 / 3600
print(f'\n  Temps estimat: ~{temps_h:.1f} hores (3 epochs)')
print('\n\u2713 Tot llest. Executa la cel\u00b7la seg\u00fcent per iniciar.')


In [ ]:
import evaluate
import numpy as np
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

metric = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds  = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = metric.compute(
        predictions=[p.strip() for p in decoded_preds],
        references=[[l.strip()] for l in decoded_labels]
    )
    return {'bleu': round(result['score'], 2)}

training_args = Seq2SeqTrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = 3,
    per_device_train_batch_size = BATCH,
    per_device_eval_batch_size  = BATCH,
    gradient_accumulation_steps = 2,
    warmup_steps                = 200,
    weight_decay                = 0.01,
    learning_rate               = 5e-5,
    fp16                        = FP16,
    predict_with_generate       = True,
    generation_max_length       = MAX_LEN,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'bleu',
    greater_is_better           = True,
    logging_steps               = 50,
    report_to                   = 'none',
    dataloader_pin_memory       = False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_tok, eval_dataset=val_tok,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f'Iniciant afinament...')
print(f'  Epochs: 3 | Batch: {BATCH} | LR: 5e-5 | FP16: {FP16}')
print('\u2500' * 50)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f'\n\u2713 Afinament completat.')
print(f'  P\u00e8rdua final: {train_result.training_loss:.4f}')
print(f'  Model guardat a: {OUTPUT_DIR}')


In [ ]:
import sacrebleu as sb
from transformers import pipeline

print('Carregant model per avaluaci\u00f3...')
translator_eval = pipeline(
    'translation',
    model=OUTPUT_DIR,
    tokenizer=OUTPUT_DIR,
    device=0 if DEVICE == 'cuda' else -1,
    max_length=MAX_LEN
)

print('Generant traduccions de validaci\u00f3...', end=' ')
translations = [
    t['translation_text']
    for t in translator_eval(val_src, batch_size=32)
]
print('fet.')

bleu = sb.corpus_bleu(translations, [val_tgt])
chrf = sb.corpus_chrf(translations, [val_tgt])

print(f'\n{"="*45}')
print(f'  RESULTATS FINALS')
print(f'{"="*45}')
print(f'  BLEU:  {bleu.score:.2f}')
print(f'  ChrF:  {chrf.score:.2f}')
print(f'{"="*45}')

print('\n5 exemples aleatoris:')
random.seed(42)
mostres = random.sample(list(zip(val_src, translations, val_tgt)), 5)
for i, (es, ca_pred, ca_ref) in enumerate(mostres):
    print(f'\n[{i+1}] ES:  {es[:100]}')
    print(f'     CA:  {ca_pred[:100]}')
    print(f'     REF: {ca_ref[:100]}')


In [ ]:
import zipfile

ZIP_PATH = '/content/model_afinar_SLPL_UV.zip' if IS_COLAB \
           else '/kaggle/working/model_afinar_SLPL_UV.zip'

print('Comprimint model...', end=' ')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in pathlib.Path(OUTPUT_DIR).rglob('*'):
        if fp.is_file():
            zf.write(fp, fp.relative_to(OUTPUT_DIR))

size_mb = os.path.getsize(ZIP_PATH) / 1024**2
print(f'fet. ({size_mb:.0f} MB)')

if IS_COLAB:
    from google.colab import files
    print('Iniciant desc\u00e0rrega...')
    files.download(ZIP_PATH)
else:
    print(f'ZIP disponible a: {ZIP_PATH}')

print('\nDesp\u00e9s de descomprimir, copia a:')
print('  C:\\Users\\santi\\OneDrive\\Documents\\SLPL\\taneu\\model-afinar\\')
